# Surrogate-guided sampling for slope-stability surrogates — results

Narrative and figures only. Every number shown here is read from `outputs/metrics/`,
written by `sgs evaluate`; nothing in this notebook fits a model or computes a metric.
To regenerate the numbers, run the pipeline (see README).

In [ ]:
# On Colab: fetch the repository; the metrics are tracked in it.
import os, subprocess, sys

if "google.colab" in sys.modules and not os.path.exists("sgs"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/bteodoru/surrogate-guided-slope-stability.git"], check=True)
    os.chdir("surrogate-guided-slope-stability")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
from IPython.display import display
from sgs import config, report

SCORING = config.ACTIVE_SCORINGS[0]

## Budget-fair arms

A, B and C are compared at their solver budgets; C0 (guided points only) is an ablation
and appears in every results table below, but not in this one.

In [ ]:
display(report.budget_frame())

## Results per test set

The two test sets are two deployment scenarios: `test-natural` samples the input space,
`test-stratified` samples the selection range of FOS uniformly.

In [ ]:
for test in config.TEST_ROLES:
    metrics = report.load_metrics(test, SCORING)
    print(f"== {test}: n = {metrics['n']}, FOS {metrics['fos_range'][0]:.2f}–{metrics['fos_range'][1]:.2f}")
    display(report.global_frame(metrics).round(3))
    for family in metrics["arms"]:
        display(report.zone_frame(metrics, family).round(3))
        report.fig_zone_rmse(metrics, family)

## Paired comparisons

Wilcoxon signed-rank on d = |e₁| − |e₂| (read as *smaller absolute errors*), with the
alternative fixed per pair in `config.COMPARISONS`, and a paired bootstrap of ΔRMSE,
which covers test-sampling variability only.

In [ ]:
for test in config.TEST_ROLES:
    display(report.comparison_frame(report.load_metrics(test, SCORING)).round(4))

## Intervals

Cross-conformal-style multiplicative intervals from nested-CV out-of-fold residuals;
coverage is reported empirically, without a formal guarantee.

In [ ]:
for test in config.TEST_ROLES:
    display(report.conformal_frame(report.load_metrics(test, SCORING)).round(3))

## Learning curve (optional analysis)

RF with fixed hyper-parameters; present only after `sgs learning-curve`.

In [ ]:
if config.learning_curve_metrics_path().exists():
    lc = report.load_learning_curve()
    for test in config.TEST_ROLES:
        report.fig_learning_curve(lc, test)